(11)=
# Chapter 11: Linear Algebra & Matrices

**Topics Covered:**
- Matrices as 2D NumPy arrays
- Matrix arithmetic and multiplication
- Solving systems of linear equations with `np.linalg.solve`
- Determinant, inverse, and rank
- Chemical engineering application: material balances

## Motivation: Why Do We Need Matrices?

Consider this simple **two-stream mixing problem** from your material-balance course:

```
Feed 1: 40 mol/s,  40% species A  ──►┐
                                     ├──► Mixed stream: F_out mol/s, 55% A
Feed 2: F2 mol/s,  80% species A  ──►┘
```

Two unknowns: **F2** and **F_out**. Two equations:

$$
\text{Overall balance:} \quad 40 + F_2 = F_\text{out}
$$

$$
\text{Species A balance:} \quad 0.40 \times 40 + 0.80\, F_2 = 0.55\, F_\text{out}
$$

With only 2 equations this is manageable by hand. But what if you had **10 streams and 10 species**? Or 50? Substitution would be hopeless.

**The key observation:** both equations are *linear* — no squares, no exponentials. That means the entire system can be written as one compact expression:

$$
\underbrace{\begin{pmatrix} 1 & -1 \\ 0.80 & -0.55 \end{pmatrix}}_{A}
\underbrace{\begin{pmatrix} F_2 \\ F_\text{out} \end{pmatrix}}_{\mathbf{x}}
=
\underbrace{\begin{pmatrix} -40 \\ -16 \end{pmatrix}}_{\mathbf{b}}
$$

**How do we solve for $\mathbf{x}$?** Multiply both sides on the left by $A^{-1}$ (the *inverse* of $A$):

$$
A^{-1} A\, \mathbf{x} = A^{-1} \mathbf{b}
\quad\Longrightarrow\quad
I\, \mathbf{x} = A^{-1} \mathbf{b}
\quad\Longrightarrow\quad
\boxed{\mathbf{x} = A^{-1} \mathbf{b}}
$$

This is the matrix equivalent of dividing both sides of a scalar equation by $a$: $ax = b \Rightarrow x = b/a$.  
In Python, `np.linalg.solve(A, b)` computes $A^{-1}\mathbf{b}$ for us — efficiently and accurately — in one call.

In [1]:
import numpy as np

# Two-stream mixer: find F2 and F_out
# Equation 1 (overall):    F2 - F_out = -40       →  coefficients: [1, -1]
# Equation 2 (species A):  0.80*F2 - 0.55*F_out = -16  →  coefficients: [0.80, -0.55]

A = np.array([[ 1.00, -1.00],
              [ 0.80, -0.55]])

b = np.array([-40.0, -16.0])

x = np.linalg.solve(A, b)
F2, F_out = x

print(f"F2    = {F2:.2f} mol/s")
print(f"F_out = {F_out:.2f} mol/s")
print(f"\nVerification: A @ x = {A @ x}  (should match b = {b})")

F2    = 24.00 mol/s
F_out = 64.00 mol/s

Verification: A @ x = [-40. -16.]  (should match b = [-40. -16.])


(11.1)=
## 11.1 Matrices as 2D NumPy Arrays

A **matrix** is just a rectangular array of numbers arranged in rows and columns. In NumPy, it is identical to a 2D array — there is no separate "matrix" type.

$$
A = \begin{pmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \\ 7 & 8 & 9 \end{pmatrix} \quad \leftarrow \text{3×3 matrix, created with } \texttt{np.array}
$$

Key terminology:

| Term | Shape | How to create |
|------|-------|---------------|
| **Row vector** | `(1, n)` — 1 row, n columns | `np.array([[1, 2, 3]])` — note the **double brackets** |
| **Column vector** | `(n, 1)` — n rows, 1 column | `np.array([[1], [2], [3]])` |
| **Square matrix** | `(n, n)` | `np.array([[...], [...]])` |
| **Identity matrix** $I$ | `(n, n)` diagonal 1s | `np.eye(n)` |

> **Note:** A plain 1D array like `np.array([1, 2, 3])` has shape `(3,)` — it is **not** a row or column vector. It has no orientation. Use double brackets `[[...]]` to make a true 2D row vector.

In [25]:
import numpy as np
# A 3×3 matrix (same as a 2D array)
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])
print("Matrix A:")
print(A)
print("Shape:", A.shape)   # (rows, cols)

Matrix A:
[[1 2 3]
 [4 5 6]
 [7 8 9]]
Shape: (3, 3)


**Three ways to store "a list of numbers" — and why the shape matters:**

In [2]:
# 1D array
v_1d = np.array([1, 2, 3])
print("\n1D array:", v_1d, "  shape:", v_1d.shape)


1D array: [1 2 3]   shape: (3,)


In [3]:
# Row vector (2D)
v_row = np.array([[1, 2, 3]])
print("\nRow vector (2D):", v_row, "  shape:", v_row.shape)


Row vector (2D): [[1 2 3]]   shape: (1, 3)


In [4]:
# Column vector (2D, explicit)
v_col = np.array([[1], [2], [3]])
print("\nColumn vector:\n", v_col, "  shape:", v_col.shape)


Column vector:
 [[1]
 [2]
 [3]]   shape: (3, 1)


In [5]:
# Identity matrix
I = np.eye(3)
print("\nIdentity matrix I:")
print(I)


Identity matrix I:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


(11.2)=
## 11.2 Matrix Arithmetic and Multiplication

Element-wise operations (`+`, `-`, `*`, `/`) work the same as with regular arrays — they act on corresponding elements.

**Matrix multiplication** is different. The `@` operator performs the true mathematical dot product of rows and columns:

$$
(AB)_{ij} = \sum_k A_{ik} B_{kj}
$$

| Operation | Symbol | Meaning |
|-----------|--------|---------|
| Element-wise add/subtract | `A + B`, `A - B` | Add matching elements |
| Element-wise multiply | `A * B` | Multiply matching elements (NOT matrix mult) |
| **Matrix multiply** | `A @ B` | Dot product (row × column) |
| Scalar multiply | `c * A` | Scale every element |
| Transpose | `A.T` | Flip rows ↔ columns |

In [16]:
A = np.array([[1, 2],
              [3, 4]])

B = np.array([[5, 6],
              [7, 8]])

print("A =\n", A)
print("B =\n", B)

A =
 [[1 2]
 [3 4]]
B =
 [[5 6]
 [7 8]]


In [17]:
# Element-wise multiply (NOT matrix multiplication)
print("\nA * B  (element-wise):\n", A * B)

# True matrix multiplication
print("\nA @ B  (matrix multiply):\n", A @ B)

# Scalar multiply
print("\n3 * A:\n", 3 * A)

# Transpose
print("\nA.T (transpose):\n", A.T)


A * B  (element-wise):
 [[ 5 12]
 [21 32]]

A @ B  (matrix multiply):
 [[19 22]
 [43 50]]

3 * A:
 [[ 3  6]
 [ 9 12]]

A.T (transpose):
 [[1 3]
 [2 4]]


In [21]:
C = A @ B

print("\nA @ B  (matrix multiply):\n", C)

I = np.eye(2)
print("\nC @ I =\n", C @ I)


A @ B  (matrix multiply):
 [[17 20]
 [46 54]]

C @ I =
 [[17. 20.]
 [46. 54.]]


(11.3)=
## 11.3 Solving Systems of Linear Equations

A system of linear equations can always be written as $A\mathbf{x} = \mathbf{b}$:

$$
\begin{cases}
2x + y = 5 \\
5x + 3y = 13
\end{cases}
\quad \Longrightarrow \quad
\underbrace{\begin{pmatrix} 2 & 1 \\ 5 & 3 \end{pmatrix}}_{A}
\underbrace{\begin{pmatrix} x \\ y \end{pmatrix}}_{\mathbf{x}}
=
\underbrace{\begin{pmatrix} 5 \\ 13 \end{pmatrix}}_{\mathbf{b}}
$$

**`np.linalg.solve(A, b)`** solves for $\mathbf{x}$ directly — no elimination by hand required.

```python
x = np.linalg.solve(A, b)
```

**Shape convention for `b`:** Even though $\mathbf{b}$ is mathematically a column vector, `np.linalg.solve` expects it as a **1D array** with shape `(n,)` — not shape `(n, 1)`. The solution `x` it returns is also 1D.

In [12]:
# Solve the system:  2x + y = 5
#                    5x + 3y = 13

A = np.array([[2, 1],
              [5, 3]])

# b is a 1D array — np.linalg.solve expects this, not a column vector
b = np.array([5, 13])
# b = np.array([[5], [13]])  # This is a 2D column vector, which will cause an error
print("b shape:", b.shape)   # (2,) — 1D, not (2,1)

# Solve for x = [x, y]
x = np.linalg.solve(A, b)
print("Solution x =", x)
print(f'x shape: {x.shape}')  # Should be (2,), not (2,1)
print(f"  x = {x[0]}")
print(f"  y = {x[1]}")

# Verify: A @ x should equal b
print("\nVerification: A @ x =", A @ x)
print("b            =", b)

b shape: (2,)
Solution x = [2. 1.]
x shape: (2,)
  x = 1.9999999999999993
  y = 1.000000000000001

Verification: A @ x = [ 5. 13.]
b            = [ 5 13]


### 11.3.1 Scaling Up: 3 Equations, 3 Unknowns

The same workflow applies for larger systems. Suppose:

$$
\begin{cases}
3x_1 + x_2 - x_3 = 2 \\
x_1 + 4x_2 + x_3 = 12 \\
2x_1 - x_2 + 5x_3 = 10
\end{cases}
$$

In [28]:
# 3×3 system
A3 = np.array([[ 3,  1, -1],
               [ 1,  4,  1],
               [ 2, -1,  5]])

b3 = np.array([2, 12, 10])

x3 = np.linalg.solve(A3, b3)
print("Solution:", x3)
print(f"  x1 = {x3[0]:.4f}")
print(f"  x2 = {x3[1]:.4f}")
print(f"  x3 = {x3[2]:.4f}")

# Verify
print("\nVerification: A @ x =", A3 @ x3)
print("b                   =", b3)

Solution: [0.63768116 2.28985507 2.20289855]
  x1 = 0.6377
  x2 = 2.2899
  x3 = 2.2029

Verification: A @ x = [ 2. 12. 10.]
b                   = [ 2 12 10]


(11.4)=
## 11.4 Determinant, Inverse, and Rank

Three important properties of square matrices:

| Property | Function | Meaning |
|----------|----------|---------|
| **Determinant** | `np.linalg.det(A)` | Scalar; $\det(A) = 0$ → no unique solution |
| **Inverse** | `np.linalg.inv(A)` | $A^{-1}$ such that $A A^{-1} = I$ |
| **Rank** | `np.linalg.matrix_rank(A)` | Number of linearly independent rows/columns |

**Key rule:** A system $A\mathbf{x} = \mathbf{b}$ has a unique solution **if and only if** $\det(A) \neq 0$ (equivalently, A has full rank).

In [13]:
A = np.array([[2, 1],
              [5, 3]])

In [14]:
# Determinant
det_A = np.linalg.det(A)
print(f"det(A) = {det_A:.4f}")
print("System has unique solution:", det_A != 0)


det(A) = 1.0000
System has unique solution: True


In [15]:
# Inverse
A_inv = np.linalg.inv(A)
print("\nA_inv =\n", A_inv)

# Verify: A @ A_inv should be the identity
print("\nA @ A_inv =\n", np.round(A @ A_inv, 10))


A_inv =
 [[ 3. -1.]
 [-5.  2.]]

A @ A_inv =
 [[1. 0.]
 [0. 1.]]


In [32]:
# Rank
rank_A = np.linalg.matrix_rank(A)
print(f"\nrank(A) = {rank_A}  (full rank: {rank_A == A.shape[0]})")


rank(A) = 2  (full rank: True)


In [16]:
B = np.array([[2, 1],
              [4, 2]])

rank_B = np.linalg.matrix_rank(B)
print(f"\nrank(B) = {rank_B}  (full rank: {rank_B == B.shape[0]})")


rank(B) = 1  (full rank: False)


In [33]:
# Singular matrix: rows are linearly dependent (row 2 = 2 × row 1)
A_singular = np.array([[1, 2],
                       [2, 4]])

print("Singular matrix A_singular =\n", A_singular)
print(f"det(A_singular) = {np.linalg.det(A_singular):.6f}")
print(f"rank(A_singular) = {np.linalg.matrix_rank(A_singular)}")

# Attempting to solve will raise an error (LinAlgError: Singular matrix)
try:
    x_sing = np.linalg.solve(A_singular, np.array([1, 2]))
    print("Solution:", x_sing)
except np.linalg.LinAlgError as e:
    print(f"\nError: {e}  ← cannot solve a singular system!")

Singular matrix A_singular =
 [[1 2]
 [2 4]]
det(A_singular) = 0.000000
rank(A_singular) = 1

Error: Singular matrix  ← cannot solve a singular system!


(11.5)=
## 11.5 Chemical Engineering Application: Material Balances

(11.5.1)=
### 11.5.1 Three-Stream Mixer-Splitter

**Steady-state material balances** on a process network are linear equations — a perfect match for `np.linalg.solve`.

Consider a simple process with three unknown flow rates ($F_1$, $F_2$, $F_3$) in mol/s:

```
F1 (A 40%)  ──►┌──────────┐
               │  Mixer   ├──► F3 ──► ┌──────────┐──► F5 (known: 80 mol/s)
F2 (A 80%)  ──►└──────────┘  (A 60%)  │ Splitter │
                                      └──────────┘──► F4 (known: 20 mol/s)
```

Balance equations (total and component):

1. **Overall balance (Mixer):** $F_1 + F_2 = F_3$
2. **Component A balance (Mixer):** $0.4 F_1 + 0.8 F_2 = 0.6 F_3$
3. **Overall balance (Splitter):** $F_3 = F_4 + F_5 = 100$ mol/s

With $F_4 = 20$ and $F_5 = 80$, we know $F_3 = 100$. The unknowns are $F_1$ and $F_2$.

Rearranging equations 1 and 2 with $F_3 = 100$:

$$
\begin{cases}
F_1 + F_2 = 100 \\
0.4 F_1 + 0.8 F_2 = 60
\end{cases}
$$

In [37]:
# Material balance on mixer
# Unknowns: F1 (mol/s), F2 (mol/s)
# Known: F3 = 100 mol/s,  x_A in feed 1 = 0.4,  x_A in feed 2 = 0.8,  x_A in F3 = 0.6

# Equations:
#   F1 + F2 = 100            (overall balance)
#   0.4*F1 + 0.8*F2 = 60    (component A balance: 0.6 * 100 = 60)

A_mat = np.array([[1.0,  1.0],
                  [0.4,  0.8]])

b_mat = np.array([100.0, 60.0])

# Solve
F = np.linalg.solve(A_mat, b_mat)
F1, F2 = F
print(f"F1 = {F1:.2f} mol/s")
print(f"F2 = {F2:.2f} mol/s")
print(f"F3 = {F1 + F2:.2f} mol/s  (check: should be 100)")

# Verify balances
print("\n--- Verification ---")
print(f"Overall balance:     F1 + F2 = {F1 + F2:.2f}  (expected 100)")
print(f"Component A balance: 0.4*F1 + 0.8*F2 = {0.4*F1 + 0.8*F2:.2f}  (expected 60)")

# Mole fraction check
x_A_F3 = (0.4*F1 + 0.8*F2) / (F1 + F2)
print(f"\nComposition of F3: x_A = {x_A_F3:.2f}  (expected 0.60)")

F1 = 50.00 mol/s
F2 = 50.00 mol/s
F3 = 100.00 mol/s  (check: should be 100)

--- Verification ---
Overall balance:     F1 + F2 = 100.00  (expected 100)
Component A balance: 0.4*F1 + 0.8*F2 = 60.00  (expected 60)

Composition of F3: x_A = 0.60  (expected 0.60)


(11.5.2)=
### 11.5.2 Energy Balance in a Heat Exchange Network

**Process Description**

Three liquid streams pass through a heat-exchange network before entering a reactor. We want to find the **unknown outlet temperatures** $T_1, T_2, T_3$ (°C).

```


                           ┌──────────────────────┐
Stream 1 (mCp=5)           │  HX-12 (UA = 2 kW/°C)│
T1,in = 100°C  ───────────►│  Heat exchange 1 ↔ 2 │──────────►  T1 (out)
                           └──────────────────────┘
                                      ▲     │
                                      │     │    
                                      │     │
                                      │     │
Stream 2 (mCp=4)                      │     │
T2,in = 60°C  ────────────────────────┘     │
                                            │
                                            │   (same stream continues)
                                            ▼
                           ┌──────────────────────┐
                           │  HX-23 (UA = 1 kW/°C)│
                           │  Heat exchange 2 ↔ 3 │──────────►  T2 (out)
                           └──────────────────────┘
                                      ▲
                                      │
Stream 3 (mCp=6)                      │
T3,in = 150°C ────────────────────────┘────────────►  T3 (out)


                    T1 (out) ─┐
                    T2 (out) ─┼──► Reactor feed (mixed or separate feeds)
                    T3 (out) ─┘

```
**Assumptions:** steady state, no heat loss, constant heat capacities, linear heat transfer between streams.


**Given Parameters**

| | Stream 1 | Stream 2 | Stream 3 |
|---|---|---|---|
| $\dot{m} C_p$ (kW/°C) | 5 | 4 | 6 |
| $T_\text{in}$ (°C) | 100 | 60 | 150 |

Heat exchanger conductances: $UA_{1\text{-}2} = 2$ kW/°C,  $UA_{2\text{-}3} = 1$ kW/°C


Step 1 — Energy Balance on Each Stream

Each stream: $\dot{m}C_p (T_\text{in} - T_\text{out}) = \sum Q_\text{exchanged}$, where $Q = UA(T_i - T_j)$.

**Stream 1:**
$$5(100 - T_1) = 2(T_1 - T_2) \quad\Longrightarrow\quad 7T_1 - 2T_2 = 500$$

**Stream 2:**
$$4(60 - T_2) + 2(T_1 - T_2) = 1(T_2 - T_3) \quad\Longrightarrow\quad 2T_1 - 7T_2 + T_3 = -240$$

**Stream 3:**
$$6(150 - T_3) + 1(T_2 - T_3) = 0 \quad\Longrightarrow\quad T_2 - 7T_3 = -900$$


Step 2 — Matrix Form $A\mathbf{T} = \mathbf{b}$

$$
\underbrace{\begin{pmatrix} 7 & -2 & 0 \\ 2 & -7 & 1 \\ 0 & 1 & -7 \end{pmatrix}}_{A}
\underbrace{\begin{pmatrix} T_1 \\ T_2 \\ T_3 \end{pmatrix}}_{\mathbf{T}}
=
\underbrace{\begin{pmatrix} 500 \\ -240 \\ -900 \end{pmatrix}}_{\mathbf{b}}
$$

In [38]:
# Heat exchange network energy balance
# Unknowns: T1, T2, T3 (outlet temperatures, °C)

A_net = np.array([[ 7, -2,  0],
                  [ 2, -7,  1],
                  [ 0,  1, -7]])

b_net = np.array([500, -240, -900], dtype=float)

print(f"det(A) = {np.linalg.det(A_net):.2f}  → unique solution exists")

T = np.linalg.solve(A_net, b_net)
print(f"\nOutlet temperatures:")
print(f"  T1 = {T[0]:.2f} °C")
print(f"  T2 = {T[1]:.2f} °C")
print(f"  T3 = {T[2]:.2f} °C")

# Verify
print(f"\nVerification: A @ T = {np.round(A_net @ T, 6)}")
print(f"b               = {b_net}")

det(A) = 308.00  → unique solution exists

Outlet temperatures:
  T1 = 94.68 °C
  T2 = 81.36 °C
  T3 = 140.19 °C

Verification: A @ T = [ 500. -240. -900.]
b               = [ 500. -240. -900.]


## Chapter 11 Summary

| Concept | Code | Notes |
|---------|------|-------|
| Create matrix | `np.array([[...], [...]])` | Same as 2D array |
| Identity matrix | `np.eye(n)` | $n \times n$ identity |
| Element-wise multiply | `A * B` | Not matrix multiply! |
| **Matrix multiply** | `A @ B` | True $AB$ product |
| Transpose | `A.T` | Swap rows ↔ cols |
| **Solve** $A\mathbf{x} = \mathbf{b}$ | `np.linalg.solve(A, b)` | ← use this for linear systems |
| Determinant | `np.linalg.det(A)` | 0 → singular, no unique solution |
| Inverse | `np.linalg.inv(A)` | Prefer `solve` over `inv @ b` |
| Rank | `np.linalg.matrix_rank(A)` | # independent rows/cols |

**Workflow for any linear system:**
1. Identify unknowns → one per column of A
2. Write one equation per row → fill in coefficients
3. Collect right-hand sides into b
4. Call `np.linalg.solve(A, b)`
5. **Always verify:** check `A @ x ≈ b`